In [1]:
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Mon Apr  6 19:12:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   30C    P0             67W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ============================================
# EXTRACT IMAGENET-100 + RESIZE TO 256px
#        (skips if already done)
# ============================================

import os, time, tarfile, shutil, subprocess
from PIL import Image

DRIVE_BASE = '/content/drive/My Drive/pe_experiment'
RESULTS_DIR = os.path.join(DRIVE_BASE, 'results')
SCRIPT_DIR = DRIVE_BASE
DATA_DIR = '/content/imagenet100_resized'
IN100_RAW = '/content/imagenet100_raw'
os.makedirs(RESULTS_DIR, exist_ok=True)

train_path = os.path.join(DATA_DIR, 'train')
val_path = os.path.join(DATA_DIR, 'val')

if (os.path.exists(train_path) and len(os.listdir(train_path)) >= 100
    and os.path.exists(val_path) and len(os.listdir(val_path)) >= 100):
    print(f'✅ ImageNet-100 already on SSD (resized)')
    print(f'  Train: {len(os.listdir(train_path))} classes')
    print(f'  Val: {len(os.listdir(val_path))} classes')
else:
    # --- Find tar files on Drive ---
    imagenet_dir = os.path.join(DRIVE_BASE, 'imagenet')
    TRAIN_TAR = None
    VAL_TAR = None
    for name in ['ILSVRC2012_img_train.tar', 'ILSVRC2012_img_train']:
        c = os.path.join(imagenet_dir, name)
        if os.path.exists(c): TRAIN_TAR = c; break
    for name in ['ILSVRC2012_img_val.tar', 'ILSVRC2012_img_val']:
        c = os.path.join(imagenet_dir, name)
        if os.path.exists(c): VAL_TAR = c; break
    print(f'Train tar: {TRAIN_TAR}')
    print(f'Val tar: {VAL_TAR}')

    # --- Download class list ---
    subprocess.run(['wget', '-q',
        'https://raw.githubusercontent.com/HobbitLong/CMC/master/imagenet100.txt',
        '-O', '/content/imagenet100.txt'], check=True)
    with open('/content/imagenet100.txt') as f:
        class_set = set(line.strip() for line in f if line.strip())
    print(f'Target classes: {len(class_set)}')

    t0 = time.time()

    # --- Extract train ---
    print(f'\nStep 1/3: Extracting train from tar...')
    raw_train = os.path.join(IN100_RAW, 'train')
    os.makedirs(raw_train, exist_ok=True)
    tf = tarfile.open(TRAIN_TAR, 'r|')
    found = 0
    for member in tf:
        if not member.name.endswith('.tar'):
            continue
        class_name = os.path.basename(member.name).replace('.tar', '')
        if class_name in class_set:
            class_dir = os.path.join(raw_train, class_name)
            os.makedirs(class_dir, exist_ok=True)
            fileobj = tf.extractfile(member)
            with tarfile.open(fileobj=fileobj, mode='r|') as ctf:
                ctf.extractall(class_dir)
            found += 1
            if found % 10 == 0:
                print(f'  {found}/100 classes ({time.time()-t0:.0f}s)')
            if found >= len(class_set):
                break
        else:
            tf.extractfile(member).read()
    tf.close()
    print(f'  Train extracted: {found} classes')

    # --- Extract val ---
    print(f'\nStep 2/3: Extracting val...')
    val_tmp = '/content/val_tmp'
    os.makedirs(val_tmp, exist_ok=True)
    with tarfile.open(VAL_TAR, 'r|') as vtf:
        vtf.extractall(val_tmp)
    subprocess.run(['wget', '-q',
        'https://raw.githubusercontent.com/tensorflow/models/master/research/slim/datasets/imagenet_2012_validation_synset_labels.txt',
        '-O', '/content/val_synsets.txt'], check=True)
    with open('/content/val_synsets.txt') as f:
        val_synsets = [line.strip() for line in f]
    raw_val = os.path.join(IN100_RAW, 'val')
    os.makedirs(raw_val, exist_ok=True)
    val_images = sorted([f for f in os.listdir(val_tmp) if f.endswith('.JPEG')])
    for img_name, synset in zip(val_images, val_synsets):
        if synset in class_set:
            class_dir = os.path.join(raw_val, synset)
            os.makedirs(class_dir, exist_ok=True)
            shutil.move(os.path.join(val_tmp, img_name), os.path.join(class_dir, img_name))
    shutil.rmtree(val_tmp)
    print(f'  Val extracted: {len(os.listdir(raw_val))} classes')

    # --- Resize all images to 256px ---
    print(f'\nStep 3/3: Resizing images to 256px...')
    t_resize = time.time()
    for split in ['train', 'val']:
        src_split = os.path.join(IN100_RAW, split)
        dst_split = os.path.join(DATA_DIR, split)
        for cls in os.listdir(src_split):
            cls_src = os.path.join(src_split, cls)
            cls_dst = os.path.join(dst_split, cls)
            os.makedirs(cls_dst, exist_ok=True)
            for img_name in os.listdir(cls_src):
                dst_path = os.path.join(cls_dst, img_name)
                if not os.path.exists(dst_path):
                    img = Image.open(os.path.join(cls_src, img_name)).convert('RGB')
                    img.thumbnail((256, 256), Image.LANCZOS)
                    img.save(dst_path, 'JPEG', quality=95)
        print(f'  {split} resized ({time.time()-t_resize:.0f}s)')

    # Clean up raw
    shutil.rmtree(IN100_RAW)

    total = time.time() - t0
    print(f'\n✅ ImageNet-100 ready in {total/60:.1f} min')
    print(f'  Train: {len(os.listdir(train_path))} classes')
    print(f'  Val: {len(os.listdir(val_path))} classes')
    !du -sh {DATA_DIR}


Train tar: /content/drive/My Drive/pe_experiment/imagenet/ILSVRC2012_img_train.tar
Val tar: /content/drive/My Drive/pe_experiment/imagenet/ILSVRC2012_img_val.tar
Target classes: 100

Step 1/3: Extracting train from tar...


/tmp/ipykernel_564/479829257.py:63: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  ctf.extractall(class_dir)


  10/100 classes (385s)
  20/100 classes (512s)
  30/100 classes (697s)
  40/100 classes (896s)
  50/100 classes (1294s)
  60/100 classes (1658s)
  70/100 classes (1843s)
  80/100 classes (2089s)
  90/100 classes (2378s)
  100/100 classes (2718s)
  Train extracted: 100 classes

Step 2/3: Extracting val...


/tmp/ipykernel_564/479829257.py:79: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  vtf.extractall(val_tmp)


  Val extracted: 100 classes

Step 3/3: Resizing images to 256px...
  train resized (645s)
  val resized (671s)

✅ ImageNet-100 ready in 59.0 min
  Train: 100 classes
  Val: 100 classes
3.3G	/content/imagenet100_resized


In [4]:
import shutil
shutil.copy2('/content/drive/My Drive/pe_experiment/full_scale_experiment.py', '/content/')
shutil.copy2('/content/drive/My Drive/pe_experiment/adversarial_pe_attacks.py', '/content/')

'/content/adversarial_pe_attacks.py'

In [5]:
!python /content/adversarial_pe_attacks.py

Device: cuda
Val: 5000 images

  LEARNED PE, seed=42
  Clean accuracy: 79.68%
  Variance Gini coefficient: 0.4556

  --- FGSM-PE ---
    ε=0.001  acc= 79.60%  Δ= +0.08pp  (8.8s)
    ε=0.005  acc= 79.34%  Δ= +0.34pp  (8.6s)
    ε=0.01   acc= 79.08%  Δ= +0.60pp  (8.6s)
    ε=0.05   acc= 77.72%  Δ= +1.96pp  (8.6s)
    ε=0.1    acc= 73.22%  Δ= +6.46pp  (8.6s)
    ε=0.2    acc= 33.40%  Δ=+46.28pp  (8.6s)
    ε=0.5    acc=  1.82%  Δ=+77.86pp  (8.6s)
    ε=1.0    acc=  1.00%  Δ=+78.68pp  (8.7s)

  --- PGD-PE (T=20) ---
    ε=0.001  acc= 79.60%  Δ= +0.08pp  (80.3s)
    ε=0.005  acc= 79.26%  Δ= +0.42pp  (80.2s)
    ε=0.01   acc= 78.90%  Δ= +0.78pp  (80.2s)
    ε=0.05   acc= 76.08%  Δ= +3.60pp  (80.3s)
    ε=0.1    acc= 68.52%  Δ=+11.16pp  (80.3s)
    ε=0.2    acc=  1.58%  Δ=+78.10pp  (80.4s)
    ε=0.5    acc=  1.22%  Δ=+78.46pp  (80.2s)
    ε=1.0    acc=  1.06%  Δ=+78.62pp  (80.3s)

  --- VTA ---
    ε=0.001  acc= 79.68%  Δ= +0.00pp  VTA_gain=0.00x  (8.6s)
    ε=0.005  acc= 79.58%  Δ= +0.10pp  